# Preprocessing comparison: raw vs cleaned

<!-- To add real photos: drop image files (.png/.jpg/.jpeg/.tif) into data/samples/ and re-run top to bottom. No code changes needed. -->

Measures the impact of `preprocessing/clean.py` by running **Tesseract** and **EasyOCR**
on the same image twice — once raw, once after the full deskew → denoise → CLAHE →
adaptive-threshold pipeline — and printing the results so raw vs cleaned is directly
comparable per engine.

**To add real photos:** drop any `.png`/`.jpg`/`.jpeg`/`.tif` files into `data/samples/`
and re-run. They will be picked up automatically.

## 1. Install

In [ ]:
!pip install -q pytesseract easyocr opencv-python pillow
!apt-get -qq update && apt-get -qq install -y tesseract-ocr

## 2. Paths and imports

In [ ]:
import glob
import os
import sys

import cv2
import numpy as np
import pytesseract
import easyocr
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

# Run from the repo root regardless of where Colab places the cwd.
if not os.path.isdir("preprocessing"):
    os.chdir("..")  # notebook is in notebooks/, repo root is one level up
assert os.path.isdir("preprocessing"), (
    "Could not find repo root. Open a terminal, cd to the repo root, then re-run."
)

# Make preprocessing/ importable as a package.
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from preprocessing.clean import preprocess  # noqa: E402

SAMPLES_DIR   = "data/samples"
PROCESSED_DIR = "data/processed"
SYNTHETIC_PATH = os.path.join(SAMPLES_DIR, "synthetic_ledger.png")

os.makedirs(SAMPLES_DIR,   exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Repo root:", os.getcwd())

## 3. Generate synthetic ledger

Same generator as `spike_ocr_test.ipynb`: ~6 data rows (date, item, qty, price, total),
mild Gaussian noise, −3° rotation to mimic a phone photo.

In [ ]:
def _load_font(size):
    candidates = [
        "/usr/share/fonts/truetype/humor-sans/Humor-Sans.ttf",
        "Humor-Sans.ttf",
        "/usr/share/fonts/truetype/comic-sans/ComicSans.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "DejaVuSans.ttf",
    ]
    for path in candidates:
        try:
            font = ImageFont.truetype(path, size)
            print(f"Using font: {path}")
            return font
        except (OSError, IOError):
            continue
    print("No TrueType font found -- falling back to PIL default.")
    return ImageFont.load_default()


def generate_synthetic_ledger(out_path, seed=42):
    """Render a ledger table, add noise + -3 deg rotation, and save to out_path."""
    rng = np.random.default_rng(seed)

    rows = [
        ["Date",       "Item",        "Qty", "Price", "Total"],
        ["2026-01-03", "Maize meal",  "2",   "3.50",  "7.00"],
        ["2026-01-05", "Cooking oil", "1",   "4.25",  "4.25"],
        ["2026-01-09", "Sugar 2kg",   "3",   "2.10",  "6.30"],
        ["2026-01-12", "Bread",       "5",   "1.15",  "5.75"],
        ["2026-01-15", "Soap bar",    "4",   "0.90",  "3.60"],
        ["2026-01-20", "Tea leaves",  "1",   "2.75",  "2.75"],
    ]

    W, H = 900, 460
    img  = Image.new("RGB", (W, H), color=(252, 250, 244))
    draw = ImageDraw.Draw(img)

    font        = _load_font(size=26)
    header_font = _load_font(size=28)
    col_x       = [30, 210, 470, 580, 720]
    y0, row_h   = 40, 56

    draw.text((30, 8), "Shop Ledger - January", fill=(20, 20, 20), font=header_font)

    for r, row in enumerate(rows):
        y          = y0 + r * row_h
        this_font  = header_font if r == 0 else font
        for c, cell in enumerate(row):
            draw.text((col_x[c], y), str(cell), fill=(15, 15, 15), font=this_font)
        draw.line([(20, y + row_h - 12), (W - 20, y + row_h - 12)],
                  fill=(200, 195, 180), width=1)

    arr   = np.asarray(img).astype(np.float32)
    noise = rng.normal(0, 8.0, arr.shape)
    arr   = np.clip(arr + noise, 0, 255).astype(np.uint8)
    img   = Image.fromarray(arr)
    img   = img.rotate(-3.0, resample=Image.BICUBIC, expand=True,
                        fillcolor=(252, 250, 244))

    img.save(out_path)
    w, h = img.size
    print(f"Saved: {out_path}  ({w}x{h})")
    return out_path


generate_synthetic_ledger(SYNTHETIC_PATH)

## 4. Run preprocessing pipeline on every sample

Calls `preprocess()` from `preprocessing/clean.py` on each image in `data/samples/` and
writes the cleaned version to `data/processed/<stem>_clean.png`.

In [ ]:
_IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}


def collect_raw_images(samples_dir):
    """Return sorted, de-duplicated list of image paths in samples_dir."""
    paths = []
    for ext in _IMAGE_EXTS:
        paths.extend(glob.glob(os.path.join(samples_dir, f"*{ext}")))
        paths.extend(glob.glob(os.path.join(samples_dir, f"*{ext.upper()}")))
    return sorted(set(paths))


raw_paths = collect_raw_images(SAMPLES_DIR)
print(f"Found {len(raw_paths)} image(s) in {SAMPLES_DIR}:")
for p in raw_paths:
    print(f"  {p}")

processed_map = {}  # raw_path -> processed_path

for raw_path in raw_paths:
    stem      = os.path.splitext(os.path.basename(raw_path))[0]
    out_path  = os.path.join(PROCESSED_DIR, f"{stem}_clean.png")
    img       = cv2.imread(raw_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"  [skip] could not read {raw_path}")
        continue
    preprocess(img, save_path=out_path)
    processed_map[raw_path] = out_path
    print(f"  [ok]   {os.path.basename(raw_path)} -> {os.path.basename(out_path)}")

print(f"\nPreprocessed {len(processed_map)} image(s).")

## 5. OCR engines

In [ ]:
# EasyOCR reader is built once — model download/init takes ~30 s on first run.
_EASYOCR_READER = None


def _get_easyocr_reader():
    global _EASYOCR_READER
    if _EASYOCR_READER is None:
        _EASYOCR_READER = easyocr.Reader(["en"], gpu=False)
    return _EASYOCR_READER


def run_tesseract(img_path):
    """Return text Tesseract recognises in img_path."""
    return pytesseract.image_to_string(Image.open(img_path)).strip()


def run_easyocr(img_path):
    """Return text EasyOCR recognises in img_path (one detection per line)."""
    results = _get_easyocr_reader().readtext(img_path, detail=0, paragraph=False)
    return "\n".join(results).strip()

## 6. Raw vs cleaned — per-engine comparison

For each source image the notebook:
1. Displays the raw image alongside the preprocessed image.
2. Prints Tesseract output: **raw** on the left, **cleaned** on the right.
3. Prints EasyOCR output: **raw** on the left, **cleaned** on the right.

Each engine's comparison is self-contained so you can focus on one without the other
getting in the way.

In [ ]:
def _side_by_side(left_title, left_text, right_title, right_text, col=50):
    """Print two text blocks in aligned columns under their titles."""
    left_lines  = (left_text  or "(no text)").splitlines() or ["(no text)"]
    right_lines = (right_text or "(no text)").splitlines() or ["(no text)"]
    n = max(len(left_lines), len(right_lines))
    left_lines  += [""] * (n - len(left_lines))
    right_lines += [""] * (n - len(right_lines))
    hdr = f"  {left_title:<{col}}| {right_title}"
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))
    for l, r in zip(left_lines, right_lines):
        print(f"  {l[:col]:<{col}}| {r}")


for raw_path, proc_path in processed_map.items():
    name = os.path.basename(raw_path)
    tag  = "(synthetic)" if os.path.abspath(raw_path) == os.path.abspath(SYNTHETIC_PATH) else "(real)"

    print("\n" + "=" * 110)
    print(f"  {name}  {tag}")
    print("=" * 110)

    # -- Visual comparison --------------------------------------------------
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(Image.open(raw_path))
    axes[0].set_title("RAW", fontsize=13)
    axes[0].axis("off")
    axes[1].imshow(Image.open(proc_path), cmap="gray")
    axes[1].set_title("CLEANED (preprocessed)", fontsize=13)
    axes[1].axis("off")
    fig.suptitle(name, fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

    # -- Tesseract: raw vs cleaned ------------------------------------------
    print("\n-- TESSERACT " + "-" * 96)
    tess_raw  = run_tesseract(raw_path)
    tess_proc = run_tesseract(proc_path)
    _side_by_side("RAW", tess_raw, "CLEANED", tess_proc)

    # -- EasyOCR: raw vs cleaned --------------------------------------------
    print("\n-- EasyOCR " + "-" * 98)
    easy_raw  = run_easyocr(raw_path)
    easy_proc = run_easyocr(proc_path)
    _side_by_side("RAW", easy_raw, "CLEANED", easy_proc)

    print()

## 7. What preprocessing changed — and what still needs work

### What to look for in the output above

**Deskew is the highest-value step for Tesseract on this image.**
The synthetic ledger is rendered at −3°. Tesseract's page-segmentation assumes horizontal
baselines: even a 2–3° tilt causes it to merge characters across adjacent columns or
introduce spurious line breaks mid-cell. Classic symptoms visible in the raw column:

- Prices like `3.50` appear as `350` or `3 50` — the decimal point is lost when
  Tesseract treats a slightly-off horizontal slice as a noise artifact.
- `5.75` can scan as `S-75` or `S75` because a skewed `5` in a light font resolves to `S`
  once the bounding box is misaligned.
- Whole rows may shift one cell to the right, causing the date column to absorb the item
  name or the quantity to land in the price column.

After deskew (and the adaptive threshold that follows), these artefacts should reduce or
disappear in the **CLEANED** column.

**CLAHE + binarisation help with paper tone but rarely fix digit errors on their own.**
The off-white background and ruled lines read cleanly after adaptive thresholding, but the
ruled lines themselves can introduce stray horizontal marks that Tesseract misreads as
underscores or dashes. If they persist in the cleaned output, try increasing the `C` offset
in `binarise()` (currently 10) to push the threshold further from the line intensity.

**EasyOCR is more tolerant of raw skew** (deep-learning features rotate internally), so
the raw-vs-cleaned delta is usually smaller for EasyOCR than for Tesseract. Where EasyOCR
does improve after preprocessing it tends to be on number accuracy in noisy regions rather
than layout structure.

### Decision checklist

| Question | What to check |
|---|---|
| Did the decimal points survive? | `3.50`, `4.25`, `2.10`, `1.15`, `0.90`, `2.75` |
| Did `5` stay `5` (not `S`)? | `5.75` total for Bread |
| Are all 6 data rows present? | Silent row drops are worse than misreads |
| Did the header row (`Date Item Qty Price Total`) survive? | |
| Did ruled lines produce junk characters? | Look for `_`, `—`, `—` in blank rows |

If errors remain after cleaning, the next steps are:
1. **Increase binarise `C`** — suppresses ruled lines.
2. **Increase denoise `h`** — stronger noise removal before thresholding.
3. **Pass `--psm 6` to Tesseract** — tells it to treat the image as a uniform block of
   text rather than a full page, which improves column alignment.
4. **Add real phone photos** — drop `.jpg`/`.png` files into `data/samples/` and re-run;
   the pipeline picks them up automatically.